# 环节 02 · KV Cache 才是内存主体（配套 Notebook）

> 配套：[benchmark.md](./benchmark.md) §八 · 导航：[环节00](./环节00-总揽与环节导航.md)
> 原理对照：[环节10 §2](../../foundation/transformer/环节10-推理解码与KV缓存详解.md)
> 定位：0.6B、522MB 的权重，Ollama 默认为什么驻留 **5.6GB**。**纯标准库。**

| 本 Notebook | 手册 | 验证什么 |
|---|---|---|
| §1 公式 | benchmark §8.1 | `2 × L × n_kv × d_h × dtype` |
| §2 对上实测 | 同节 | 112 KiB/token；40960 ctx → 4.5 GiB KV |
| §3 权重 vs KV | 陷阱 2 | 先定 ctx 再比「谁省内存」 |
| §4 档位矩阵 | §8.3 | 16–128GB 统一内存怎么选模型 |
| §5 GQA 怎么砍 | 环节10 / 环节11 | KV 头数减半，Cache 减半 |


## 1. 每 token 的 KV 字节

自回归每走一步都要缓存该层每个 KV 头的 Key 与 Value：

```
每 token KV 字节 = 2(K/V) × n_layers × n_kv_heads × head_dim × dtype字节
总 KV            = 每 token KV × 上下文长度
```

Qwen3-0.6B：28 层 / 8 个 KV 头 / head_dim 128 / fp16。


In [ ]:
def kv_bytes_per_token(n_layers, n_kv_heads, head_dim, dtype_bytes=2) -> int:
    return 2 * n_layers * n_kv_heads * head_dim * dtype_bytes


# Qwen3-0.6B（benchmark.md §8.1）
PER = kv_bytes_per_token(28, 8, 128, 2)
print(f"2 × 28 × 8 × 128 × 2 = {PER} B/token = {PER/1024:.0f} KiB/token")

# 实测：num_ctx 1024→4096，ollama ps SIZE 671MB→1.0GB
delta_ctx = 4096 - 1024
delta_gb = 1.0 - 0.671
measured = delta_gb * (1024**3) / delta_ctx
print(f"实测 (1.0-0.671) GB / {delta_ctx} tok = {measured/1024:.0f} KiB/token")
print("公式与实测对齐。")


## 2. 把上下文拉开：KV 怎样把 522MB 模型吃成 5.6GB


In [ ]:
WEIGHT_GB = 0.52  # qwen3:0.6b Q4 体积
OVERHEAD_GB = 0.6  # 运行时 / 碎片，经验值


def resident_gb(ctx: int) -> float:
    return WEIGHT_GB + PER * ctx / (1024**3) + OVERHEAD_GB


print(f"{'num_ctx':>10} {'KV GiB':>10} {'预估驻留':>10} {'手册实测':>10}")
for ctx, measured in [(1024, 0.671), (4096, 1.0), (40960, 5.6)]:
    kv = PER * ctx / (1024**3)
    pred = resident_gb(ctx)
    print(f"{ctx:>10,} {kv:>10.2f} {pred:>10.2f} {measured:>10.2f}")

kv_40k = PER * 40960 / (1024**3)
print(f"\n40960 ctx 的 KV 单独就是 {kv_40k:.1f} GiB")
print(f"占 5.6GB 驻留的 {kv_40k/5.6:.0%} —— 权重只是零头。")
print("\nOllama 默认按 VRAM 档位拉满上下文：48GB 机型上 0.6B 直接给你 40960。")
print("任何内存对比前先看 ollama ps 的 CONTEXT，并显式设 num_ctx。")


## 3. 「谁省内存」比的常常是上下文预算，不是引擎


In [ ]:
print("同一 0.6B、同一台机器：")
print(f"  Ollama 默认 ctx=40960  →  5.6 GB")
print(f"  Ollama 显式 ctx=4096   →  1.0 GB")
print(f"  MLX 4-bit 峰值         →  0.92 GB  （benchmark 短上下文）")
print()
print("拿 5.6GB 去和 0.92GB 比『Ollama 更吃内存』，比的是 ctx，不是引擎。")
print("锁 ctx 再比：1.0 vs 0.92，差距主要来自量化算法（Q4_K_M 517MB vs MLX 4-bit 336MB）。")


## 4. Mac 统一内存档位：先算 KV，再选模型

预算 ≈ 权重 + KV + 系统。本机 Metal `recommendedMaxWorkingSetSize ≈ 40 GB`（48GB 机），超了就开始换页、速度断崖。

Q4 体积经验：1B 以下按 **0.85 GB/B**，7B–30B 按 **0.7 GB/B**。换机型只改 `UNIFIED_GB`。


In [ ]:
UNIFIED_GB = 48  # 本机；16 / 24 / 36 / 64 / 128 按你的统一内存改


def q4_weight_gb(params_b: float) -> float:
    return params_b * (0.85 if params_b < 1 else 0.70)


def plan(unified_gb: int, params_b: float, n_layers, n_kv, d_h, ctx, sys_frac=0.25):
    weight = q4_weight_gb(params_b)
    per = kv_bytes_per_token(n_layers, n_kv, d_h)
    kv = per * ctx / (1024**3)
    need = weight + kv
    budget = unified_gb * (1 - sys_frac)
    ok = need <= budget
    return weight, kv, need, budget, ok


print(f"本机统一内存 {UNIFIED_GB} GB，系统预留 25% 后预算 {UNIFIED_GB * 0.75:.0f} GB\n")
print(f"{'统一内存':>8} {'模型':>8} {'ctx':>7} {'权重':>7} {'KV':>7} {'合计':>7} {'预算':>7} 判定")
CASES = [
    (16, 0.6, 28, 8, 128, 4096, "0.6B"),
    (16, 8, 32, 8, 128, 8192, "8B GQA"),
    (24, 14, 40, 8, 128, 8192, "14B"),
    (UNIFIED_GB, 9, 32, 8, 128, 4096, "9B 短ctx"),
    (UNIFIED_GB, 27, 48, 8, 128, 8192, "27B"),
    (64, 70, 80, 8, 128, 8192, "70B Q4"),
]
for uni, pb, L, kvh, dh, ctx, tag in CASES:
    w, kv, need, budget, ok = plan(uni, pb, L, kvh, dh, ctx)
    flag = "OK" if ok else "要换页/降档"
    print(f"{uni:>6}GB {tag:>8} {ctx:>7,} {w:>6.1f}G {kv:>6.1f}G {need:>6.1f}G {budget:>6.1f}G  {flag}")

print("\n上面 9B 短 ctx 是通用 GQA 估算。本机实测不要硬套层数：")
print("  qwen3.5:9b  ctx 4096 → 5.5 GB；默认 262144 → 15 GB；KV ≈ 9.5 GB")
print("  （小跨度 4096→16384 只从 5.5 涨到 6.0，会被 ollama ps 的 0.1GB 精度淹没）")
print("先定上下文，再决定能量到多大的模型——别倒过来。")


## 5. GQA / MLA：砍的是 KV 头数，不是层数


In [ ]:
# 同为 32 层、head_dim 128、fp16、ctx=8192
ctx = 8192
print(f"{'注意力':<16} {'KV头':>6} {'KiB/tok':>10} {'8k ctx':>10}")
for name, kvh in [("MHA", 32), ("GQA-8", 8), ("GQA-4", 4), ("MQA", 1)]:
    per = kv_bytes_per_token(32, kvh, 128)
    tot = per * ctx / (1024**3)
    print(f"{name:<16} {kvh:>6} {per/1024:>10.0f} {tot:>9.2f}G")

print("\nMLA（DeepSeek）把每层 K/V 压成潜向量，entry size 再降一档——")
print("那是模型架构的事，见 foundation/transformer 与 model-cases/deepseek，")
print("本地 GGUF 跑的仍是『量化后的这份 KV』。")
